In [1]:
import asyncio
import json
import os
import random
import time
from pathlib import Path

import dotenv
from tqdm.auto import tqdm

from financial_qa.headers.agent_loop import HeadersAgentLoop
from financial_qa.headers.headers_index import HeadersIndex
from financial_qa.evaluation import evaluate_async, load_jsonl

In [2]:
dotenv.load_dotenv('.env')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY is required')
OMC_API_KEY=os.getenv('OMC_API_KEY')

In [3]:
DATASET_FILE   = 'data/dataset.jsonl'
DATASET_SPLIT  = None          # None = all splits
MAX_QUESTIONS  = None
RANDOM_SEED    = 67

STORE_NAME     = 'default'
# GEN_MODEL      = 'google/gemini-3.1-flash-lite-preview'
GEN_MODEL      = 'Anthropic/opus-4.7'
MAX_TURNS      = 10
QUERY_CONCURRENCY = 20

JUDGE_MODEL    = 'google/gemini-2.0-flash-lite-001'

## Load dataset

In [4]:
all_records = list(load_jsonl(DATASET_FILE).values())

if DATASET_SPLIT is not None:
    all_records = [r for r in all_records if r.get('split') == DATASET_SPLIT]

random.seed(RANDOM_SEED)
records = all_records if MAX_QUESTIONS is None else random.sample(all_records, MAX_QUESTIONS)
golden  = {r['question_id']: r for r in records}

print(f'Loaded {len(records)} questions (split={DATASET_SPLIT!r}, seed={RANDOM_SEED})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))

Loaded 449 questions (split=None, seed=67)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


## Init agent

In [5]:
index = HeadersIndex(name=STORE_NAME)
print(index.get_structure())

data/parsed/  (headers index: default)
  alfa/2022/annual_parsed.md [indexed]
  alfa/2023/annual_parsed.md [indexed]
  alfa/2024/annual_parsed.md [indexed]
  alfa/2025/annual_parsed.md [indexed]
  domrf/2022/annual_parsed.md [indexed]
  domrf/2023/6m_parsed.md [indexed]
  domrf/2023/annual_parsed.md [indexed]
  domrf/2024/6m_parsed.md [indexed]
  domrf/2024/annual_parsed.md [indexed]
  domrf/2025/6m_parsed.md [indexed]
  domrf/2025/annual_parsed.md [indexed]
  gpb/2022/annual_parsed.md [indexed]
  gpb/2023/annual_parsed.md [indexed]
  gpb/2024/annual_parsed.md [indexed]
  gpb/2025/annual_parsed.md [indexed]
  mkb/2022/annual_parsed.md [indexed]
  mkb/2023/6m_parsed.md [indexed]
  mkb/2023/annual_parsed.md [indexed]
  mkb/2024/6m_parsed.md [indexed]
  mkb/2024/annual_parsed.md [indexed]
  mkb/2025/6m_parsed.md [indexed]
  mkb/2025/annual_parsed.md [indexed]
  rshb/2022/annual_parsed.md [indexed]
  rshb/2023/6m_parsed.md [indexed]
  rshb/2023/annual_parsed.md [indexed]
  rshb/2024/6m_par

In [6]:
loop = HeadersAgentLoop(
    index=index,
    model=GEN_MODEL,
    max_turns=MAX_TURNS,
    api_key=OMC_API_KEY,
    log_dir='logs/runs',
    base_url="https://api.ohmycode.ai/v1/"
)

## Run queries

In [7]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'], query_id=rec['question_id'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        return {
            'question_id': rec['question_id'],
            'question':    rec['question'],
            'answer':      answer,
            'evidence':    [],
            'confidence':  confidence,
            'error':       error,
            'elapsed_s':   time.perf_counter() - start,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = [asyncio.create_task(_bound(rec)) for rec in records]
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='q')
    for coro in asyncio.as_completed(tasks):
        result = await coro
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.1f}",
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')

Querying agent:   0%|          | 0/449 [00:00<?, ?q/s]

Done: 449 answers, 7 errors


In [8]:
if query_errors:
    print(f'{len(query_errors)} errors:')
    for e in query_errors:
        print(f'  [{e["question_id"]}] {e["error"]}')

7 errors:
  [q_deda60af1461b23a] Server disconnected
  [q_03cb5e036ae4eb77] Server disconnected
  [q_130b2df886f48082] Server disconnected
  [q_341836d99a0709a8] Server disconnected
  [q_6310c758836c1b68] Server disconnected
  [q_4a242d8e016cf078] Server disconnected
  [q_865e5a8749686e79] Server disconnected


## LLM-as-judge

In [9]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    api_key=OPENROUTER_API_KEY,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    progress_desc='LLM-as-judge',
)

accuracy = result['correct'] / result['total']
print(f"Accuracy: {result['correct']}/{result['total']} = {accuracy:.1%}")

LLM-as-judge: 100%|██████████| 449/449 [00:03<00:00, 140.08question/s, accuracy=86.64%, correct=389, errors=0]


Accuracy: 389/449 = 86.6%


## Results

In [10]:
for res in result['results']:
    mark = '✓' if res['judge_score'] == 1 else '✗'
    print(f"{mark} [{res['question_id']}]")
    print(f"  Q:    {res['question']}")
    print(f"  Gold: {res['gold_answer']}")
    print(f"  Pred: {res['predicted_answer']}")
    print(f"  Why:  {res['judge_reasoning']}")
    print()

✓ [q_009c97884b8dc010]
  Q:    Каковы чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года?
  Gold: 5 592 млн руб.
  Pred: Чистые комиссионные доходы Банка ДОМ.РФ за шесть месяцев, закончившихся 30 июня 2025 года, составили **5 592 млн руб.**
  Why:  Оба ответа указывают на одинаковую сумму чистых комиссионных доходов.

✓ [q_00d660efcf3e4607]
  Q:    Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?
  Gold: 1,151 тыс. белорусских рублей
  Pred: 1 151 тыс. бел. руб. (Доллар США LIBOR — 141; Евро LIBOR — 1 010).
  Why:  Оба ответа указывают на одну и ту же сумму, хотя в предсказанном ответе также указаны суммы для других валют.

✓ [q_00e8ee23542080b4]
  Q:    Какие макроэкономические факторы используются ВТБ в качестве базового набора для учёта макроэкономических ожиданий при расчёте PD в течение 12 месяцев согласно отчётности за 2025 год?


In [11]:
wrong = [r for r in result['results'] if r['judge_score'] != 1]
print(f"Wrong answers ({len(wrong)}/{result['total']}):\n")
for res in wrong:
    print(f"  [{res['question_id']}] {res['question']}")
    print(f"    Gold: {res['gold_answer']}")
    print(f"    Pred: {res['predicted_answer']}")
    print()

Wrong answers (60/449):

  [q_03cb5e036ae4eb77] Каков общий объём средств клиентов ВТБ по состоянию на 31 декабря 2024 года согласно обобщённой консолидированной финансовой отчётности?
    Gold: 26 926,8 млрд руб.
    Pred: 

  [q_0678360197cb8600] Какова доля государственных облигаций в валовой балансовой стоимости инвестиций в долговые ценные бумаги ЗАО «Альфа-Банк» на 31 декабря 2025 года?
    Gold: Государственные облигации составляют 153,393 тыс. BYR из общей валовой балансовой стоимости инвестиций в долговые ценные бумаги 157,712 тыс. BYR, что соответствует приблизительно 97,3%.
    Pred: На основании полученных данных точно ответить на вопрос невозможно: в извлечённых разделах представлена только общая сумма инвестиций в долговые ценные бумаги ЗАО «Альфа-Банк» на 31 декабря 2025 года — 157 706 тыс. бел. руб. (Примечание 8), однако детальная разбивка валовой балансовой стоимости по типам эмитентов (в том числе доля государственных облигаций) в полученных сегментах отсутствует.

 